# Primera versión de la actividad

En esta versión, empaquetamos toda la preparación y entrenamiento del modelo, así como el guardado en disco dentro de una sola función, usando después el modelo resultado con otra fución definida para tal fin.

In [2]:
### TRAIN
def train_estimator():

    import pandas as pd
    from sklearn.feature_extraction.text import TfidfTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.metrics import accuracy_score, balanced_accuracy_score
    # import os
    import pickle

    #
    # Manejo de la data
    #
    dataframe = pd.read_csv(
        "../files/input/sentences.csv.zip",
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase
    target = dataframe.target
    
    X_train, X_test, y_train, y_test = train_test_split(
        data,
        target,
        test_size=0.3,
        shuffle=False,
    )

    #
    # Modelo de regresión logística
    #
    vectorizer = CountVectorizer(
        lowercase=True,
        analyzer="word",
        token_pattern=r"\b[a-zA-Z]\w+\b",
        stop_words="english",
    )

    transformer = TfidfTransformer()

    lr_estimator = Pipeline(
        steps=[
            ("vectorizer", vectorizer),
            ("transformer", transformer),
            ("estimator", LogisticRegression(max_iter=1000)),
        ],
        verbose=False,
    )

    lr_estimator.fit(X_train, y_train)

    with open("estimator.pickle", "wb") as file:
        pickle.dump(lr_estimator, file)

train_estimator()

In [3]:
# USE
def use_estimator():

    import pickle

    import pandas as pd

    dataframe = pd.read_csv(
        "../files/input/sentences.csv.zip",
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase

    with open("estimator.pickle", "rb") as file:
        estimator = pickle.load(file)

    prediction = estimator.predict(data)

    return prediction


use_estimator()

array(['neutral', 'positive', 'positive', ..., 'neutral', 'positive',
       'neutral'], shape=(2264,), dtype=object)

Aquí podemos notar la ventaja de emplear **pipelines**, los cuales pueden convertir la serie de pasos en una estructura de tipo modelo, que aplica cada uno de los elementos (transformaciones, entrenamiento, etc.) al conjunto de datos de entrenamiento que se proporciona como entrada.

De esta manera, no nos tenemos que preocupar por guardar por aparte los transformadores y el modelo, teniendo todo en el mismo archivo y simplificando el proceso de guardado y uso.